# Fine-tune Gemma 4 E2B for medical text simplification

**Setup — Kaggle Notebook Settings:**
- Accelerator: **GPU T4 x2** (or T4 x1 if quota is tight)
- Internet: **On** (needed to pull base model from HF)
- Persistence: **Files only**
- Add dataset: **alio-medical-training-data** (containing `train.jsonl` and `val.jsonl`)

Outputs (downloadable from the Notebook's Output tab when done):
- `lora_adapter/` — LoRA adapter (~50 MB)
- `gemma4-e2b-medical-q4_k_m.gguf` — quantized GGUF for Ollama
- `Modelfile` — drop next to GGUF and `ollama create alio-medical -f Modelfile`

## 1. Install Unsloth + deps

In [ ]:
%%capture
# Kaggle's environment is curated for Unsloth — single install command
!pip install --upgrade "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

## 2. Verify GPU + imports

In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
print(f'torch={torch.__version__}, cuda={torch.cuda.is_available()}, gpu={torch.cuda.get_device_name(0)}')
print(f'VRAM={torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 3. Load base Gemma 4 E2B (Unsloth's 4-bit version)

In [ ]:
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3n-E2B-it",  # Unsloth's pre-quantized 4-bit; swap to gemma-4-e2b-it when published
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # auto bf16 on Ampere+, fp16 on Turing
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Load and format training data

In [ ]:
import os

# Locate the JSONL files — Kaggle mounts datasets under /kaggle/input/
DATA_DIR = '/kaggle/input/alio-medical-training-data'
if not os.path.exists(DATA_DIR):
    # Fallback: search any input dir for the JSONLs
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.jsonl' in files:
            DATA_DIR = root
            break
print('Loading from:', DATA_DIR)

train_ds = load_dataset('json', data_files=os.path.join(DATA_DIR, 'train.jsonl'))['train']
val_ds   = load_dataset('json', data_files=os.path.join(DATA_DIR, 'val.jsonl'))['train']
print(f'train: {len(train_ds)}, val: {len(val_ds)}')

# ShareGPT-style rows -> chat template
def format_chat(example):
    role_map = {'system': 'system', 'human': 'user', 'gpt': 'assistant'}
    messages = [
        {'role': role_map[t['from']], 'content': t['value']}
        for t in example['conversations']
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

train_ds = train_ds.map(format_chat)
val_ds   = val_ds.map(format_chat)

print('--- sample formatted example ---')
print(train_ds[0]['text'][:500])

## 5. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir='outputs',
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        eval_strategy='steps',
        eval_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to='none',
    ),
)

trainer.train()

## 6. Save LoRA adapter

In [ ]:
model.save_pretrained('/kaggle/working/lora_adapter')
tokenizer.save_pretrained('/kaggle/working/lora_adapter')
print('Saved adapter')
!ls -la /kaggle/working/lora_adapter

## 7. Export to GGUF (q4_k_m) for Ollama

In [ ]:
model.save_pretrained_gguf(
    '/kaggle/working/gguf',
    tokenizer,
    quantization_method='q4_k_m',
)

# Write a Modelfile so the local Ollama side is one command
modelfile = '''FROM ./Modelfile.gguf

PARAMETER temperature 0.4
PARAMETER top_p 0.9
PARAMETER stop "<end_of_turn>"
PARAMETER num_ctx 2048

SYSTEM "You are a medical assistant that explains health information in plain language for family members. Always reply in JSON when the prompt requests structured output."
'''
with open('/kaggle/working/Modelfile', 'w') as f:
    f.write(modelfile)

!ls -la /kaggle/working/gguf /kaggle/working/Modelfile

## 8. Quick smoke test

Verify the fine-tuned model produces sensible output before you download it.

In [ ]:
FastLanguageModel.for_inference(model)
test_input = '''Lab Results: Comprehensive Metabolic Panel
Collected on May 14, 2026

  Sodium: 138 mmol/L  (Normal range: 135-145)
  Glucose: 94 mg/dL  (Normal range: 65-99)
  Albumin: 5.3 g/dL  (Normal range: 3.5-5.0) High'''

messages = [
    {'role': 'system', 'content': 'You are a medical assistant. Reply in JSON.'},
    {'role': 'user', 'content': test_input},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.4)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## After this finishes

1. In the Kaggle Notebook **Output** tab, download:
   - The entire `gguf/` folder (contains the `.gguf` file + tokenizer files)
   - The `Modelfile`
2. On your local machine (after Ollama is running):
   ```powershell
   cd path\to\downloaded\gguf
   ollama create alio-medical -f ..\Modelfile
   ollama run alio-medical
   ```
3. In your app: `$env:USE_LOCAL_OLLAMA="1"; $env:OLLAMA_MODEL="alio-medical"`